# AgniDrishti photo model: YOLO11n on D-Fire

Trains the field-photo model from spec §11.1: YOLO11n, 640 px, about 30 epochs, classes smoke and fire.

Before running:
1. **Add Input**: dataset `sayedgamal99/smoke-fire-detection-yolo` (D-Fire in YOLO format, with train/val/test).
2. **Settings**: Accelerator = GPU (P100 or T4), Internet = On.
3. **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab. About 2–3 hours.
4. When it finishes, open the version's **Output** tab and download `dfire_yolo11n_best.pt`. Put it in `data/models/` in the repo.

In [ ]:
!pip install -q ultralytics

In [ ]:
import glob, os

# Find <split>/images folders that have a sibling <split>/labels folder.
splits = {}
for img in glob.glob('/kaggle/input/**/images', recursive=True):
    name = os.path.basename(os.path.dirname(img)).lower()
    if os.path.isdir(os.path.join(os.path.dirname(img), 'labels')):
        splits[{'valid': 'val', 'validation': 'val'}.get(name, name)] = img
print(splits)
assert 'train' in splits and ('val' in splits or 'test' in splits), 'attach the D-Fire dataset (see the first cell)'

# D-Fire uses 0 = smoke, 1 = fire and has more fire boxes (14,692) than smoke boxes (11,865).
counts = {}
for f in glob.glob(os.path.join(os.path.dirname(splits['train']), 'labels', '*.txt')):
    for line in open(f):
        if line.strip():
            counts[line.split()[0]] = counts.get(line.split()[0], 0) + 1
print('train boxes per class id:', counts)
if counts.get('0', 0) > counts.get('1', 0):
    print('WARNING: class 0 has more boxes than class 1; check the dataset uses 0 = smoke, 1 = fire')

val = splits.get('val', splits.get('test'))
open('/kaggle/working/dfire.yaml', 'w').write(
    f"train: {splits['train']}\nval: {val}\n"
    + (f"test: {splits['test']}\n" if 'test' in splits else '')
    + "names:\n  0: smoke\n  1: fire\n")
print(open('/kaggle/working/dfire.yaml').read())

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(data='/kaggle/working/dfire.yaml', imgsz=640, epochs=30, batch=32, patience=10,
            project='/kaggle/working/runs', name='dfire_yolo11n', exist_ok=True)

In [ ]:
import shutil

best = YOLO('/kaggle/working/runs/dfire_yolo11n/weights/best.pt')
m = best.val(data='/kaggle/working/dfire.yaml', split='test' if 'test' in splits else 'val', imgsz=640)
for i, name in best.names.items():
    print(f'{name}: mAP50 {m.box.ap50[i]:.3f}  mAP50-95 {m.box.ap[i]:.3f}')
print(f'all: mAP50 {m.box.map50:.3f}')
shutil.copy('/kaggle/working/runs/dfire_yolo11n/weights/best.pt', '/kaggle/working/dfire_yolo11n_best.pt')
print('download /kaggle/working/dfire_yolo11n_best.pt from the Output tab')